# ✨ Image Enhancer — Real-ESRGAN + GFPGAN (free, on Colab's GPU)

Upscale images **2x/4x** and optionally restore faces, using the same open-source engine behind tools like Upscayl.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save` (this is the important step).

Then run the cells top to bottom:
1. Check the GPU
2. Install the models (~2 min, once per session)
3. Upload your images
4. Enhance
5. Download the results

Tips: set `MODEL = 'anime'` in Step 4 for illustrations, or `FACE_ENHANCE = False` for landscapes and objects.

In [ ]:
# Step 1 — Check the GPU (you should see "Tesla T4" below)
!nvidia-smi

In [ ]:
# Step 2 — Install Real-ESRGAN + GFPGAN (~2 minutes, run once per session)
!pip install -q realesrgan gfpgan basicsr facexlib

# Patch basicsr for newer torchvision (the functional_tensor module was renamed)
import glob
for f in glob.glob('/usr/local/lib/python3*/dist-packages/basicsr/data/degradations.py'):
    s = open(f).read().replace(
        'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
        'from torchvision.transforms.functional import rgb_to_grayscale')
    open(f, 'w').write(s)

print('✅ Installed and ready')

In [ ]:
# Step 3 — Upload your images (you can select several at once)
import os
from google.colab import files

os.makedirs('inputs', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join('inputs', name), 'wb') as f:
        f.write(data)

print(f'✅ {len(uploaded)} image(s) ready in inputs/')

In [ ]:
# Step 4 — Enhance
MODEL = 'photo'        # 'photo' for real photos, 'anime' for illustrations/drawings
SCALE = 4              # 2 or 4
FACE_ENHANCE = True    # great for portraits; set False for landscapes and objects

import os, cv2, torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

if MODEL == 'anime':
    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
    weights = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth'
else:
    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    weights = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'

# tile=400 keeps memory in check on the free T4 GPU, even for large images
upsampler = RealESRGANer(scale=4, model_path=weights, model=net,
                         tile=400, tile_pad=10, pre_pad=0,
                         half=torch.cuda.is_available())

face_enhancer = None
if FACE_ENHANCE:
    from gfpgan import GFPGANer
    face_enhancer = GFPGANer(
        model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
        upscale=SCALE, arch='clean', channel_multiplier=2, bg_upsampler=upsampler)

for name in sorted(os.listdir('inputs')):
    img = cv2.imread(os.path.join('inputs', name), cv2.IMREAD_COLOR)
    if img is None:
        print(f'⏭️ Skipping {name} (not a readable image)')
        continue
    if face_enhancer is not None:
        _, _, output = face_enhancer.enhance(img, has_aligned=False,
                                             only_center_face=False, paste_back=True)
    else:
        output, _ = upsampler.enhance(img, outscale=SCALE)
    out_name = os.path.splitext(name)[0] + f'_enhanced_x{SCALE}.png'
    cv2.imwrite(os.path.join('outputs', out_name), output)
    print(f'✅ {name} → outputs/{out_name}')

print('\nDone — run Step 5 to download.')

In [ ]:
# Step 5 — Download the results as a zip
from google.colab import files
!zip -q -j enhanced_images.zip outputs/*
files.download('enhanced_images.zip')